# 1. Import das bibliotecas necessárias

In [12]:
%pip install scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import joblib


Note: you may need to restart the kernel to use updated packages.


# 2. Carregamento dos dados

In [13]:
# Carregando o dataset para o repositório

path = '../data/raw/Telco_customer_churn.xlsx'

df = pd.read_excel(path)

In [14]:
df.rename(columns={
            'CustomerID': 'customer_id',
            'Count': 'count',	
            'Country': 'country',	
            'State': 'state',
            'City': 'city',
            'Zip Code': 'zip_code',
            'Lat Long': 'lat_long',
            'Latitude': 'latitude',
            'Longitude': 'longitude',	
            'Gender': 'gender',
            'Senior Citizen': 'senior_citizen',
            'Partner': 'partner',	
            'Dependents': 'dependents',	
            'Tenure Months': 'tenure_months',	
            'Phone Service': 'phone_service',	
            'Multiple Lines': 'multiple_lines',	
            'Internet Service': 'internet_service',	
            'Online Security': 'online_security',	
            'Online Backup': 'online_backup',	
            'Device Protection': 'device_protection',	
            'Tech Support': 'tech_support',	
            'Streaming TV': 'streaming_tv',	
            'Streaming Movies': 'streaming_movies',	
            'Contract': 'contract',	
            'Paperless Billing': 'paperless_billing',	
            'Payment Method': 'payment_method',
            'Monthly Charges': 'monthly_charges',	
            'Total Charges': 'total_charges',	
            'Churn Label': 'churn_label',	
            'Churn Value': 'churn_value',	
            'Churn Score': 'churn_score',	
            'CLTV': 'cltv',	
            'Churn Reason': 'churn_reason'},
            inplace=True
)

# Tratamento de One hot encoding

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        7043 non-null   object 
 1   count              7043 non-null   int64  
 2   country            7043 non-null   object 
 3   state              7043 non-null   object 
 4   city               7043 non-null   object 
 5   zip_code           7043 non-null   int64  
 6   lat_long           7043 non-null   object 
 7   latitude           7043 non-null   float64
 8   longitude          7043 non-null   float64
 9   gender             7043 non-null   object 
 10  senior_citizen     7043 non-null   object 
 11  partner            7043 non-null   object 
 12  dependents         7043 non-null   object 
 13  tenure_months      7043 non-null   int64  
 14  phone_service      7043 non-null   object 
 15  multiple_lines     7043 non-null   object 
 16  internet_service   7043 

In [ ]:
df_model = df.drop(columns=[
    'customer_id',
    'count',
    'country',
    'state',
    'city',
    'zip_code',
    'lat_long',
    'latitude',
    'longitude',
    'churn_label',
    'churn_score',
    'cltv',
    'churn_reason'
])


In [17]:
#categorical_cols = columns=['country','state','city','lat_long','gender','senior_citizen','partner','dependents','phone_service','multiple_lines','internet_service','device_protection','online_security','online_backup']
categorical_cols = df_model.select_dtypes(include='object').columns
df_encoded = pd.get_dummies(df_model,columns=categorical_cols, drop_first=True)

print(f"Shape pós enconding: {df_encoded.shape}")
df_encoded.head()

Shape pós enconding: (7043, 6560)


,tenure_months,monthly_charges,churn_value,gender_Male,senior_citizen_Yes,partner_Yes,dependents_Yes,phone_service_Yes,multiple_lines_No phone service,multiple_lines_Yes,...,total_charges_8496.7,total_charges_8529.5,total_charges_8543.25,total_charges_8547.15,total_charges_8564.75,total_charges_8594.4,total_charges_8670.1,total_charges_8672.45,total_charges_8684.8,total_charges_
0,2,53.85,1,True,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2,70.70,1,False,False,False,True,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2,8,99.65,1,False,False,False,True,True,False,True,...,False,False,False,False,False,False,False,False,False,False
3,28,104.80,1,False,False,True,True,True,False,True,...,False,False,False,False,False,False,False,False,False,False
4,49,103.70,1,True,False,False,True,True,False,True,...,False,False,False,False,False,False,False,False,False,False


# Experimentação e MVP (regressão logística - baseline)

In [18]:
from sklearn.model_selection import train_test_split

# Divida o dataset em features (X) e target (y)
X = df_encoded.drop(columns=['churn_value'])
y = df_encoded['churn_value']

# Divida em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [19]:
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

#mlflow.set_experiment("telco_customer_churn")

#with mlflow.start_run(run_name="telco_customer_churn"):
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test)
test_precision = precision_score(y_test, y_pred_test)
test_recall = recall_score(y_test, y_pred_test)

    # mlflow.log_metric("train_accuracy", train_accuracy)
    # mlflow.log_metric("test_accuracy", test_accuracy)
    # mlflow.log_metric("test_f1_score", test_f1)
    # mlflow.log_metric("test_precision", test_precision)
    # mlflow.log_metric("test_recall", test_recall)

overfitting = train_accuracy - test_accuracy
    # mlflow.log_metric("overfitting", overfitting)

    # mlflow.sklearn.log_model(model, "model")

print(f"=== LOGISTIC REGRESSION ===")
print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy:  {test_accuracy:.4f}")
print(f"Test F1 Score:  {test_f1:.4f}")

print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall:    {test_recall:.4f}")
print(f"Overfitting:    {overfitting:.4f}")

=== LOGISTIC REGRESSION ===
Train Accuracy: 0.8829
Test Accuracy:  0.7999
Test F1 Score:  0.6006
Test Precision: 0.6386
Test Recall:    0.5668
Overfitting:    0.0830


In [20]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

resultados = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring=['accuracy','precision','recall','f1']
)

KeyboardInterrupt: 